## **서울시 따릉이 이용정보 전처리**

- 2018 ~ 2025 서울시 공공자전거(따릉이) 이용정보 종합
  
- PM 정책 효과 분석을 위한 '월별, 자치구별, 대여소별 누적 이용량' 계산
  
- 원본 데이터의 행 수는 수백만 개 이상으로 방대한 양의 데이터
  
- `연월, 대여소번호`를 기준으로 `groupby` 작업을 수행하여 데이터 크기 압축

- 미리 만들어둔 `통합_따릉이_대여소_마스터.csv`와 병합하면서 하나의 통합 파일 생성

- `df_merged` 데이터 점검, CSV 저장 (utf-8-sig)

#### **결과물: 따릉이_월별_이용정보_통합.csv**

  #### **목표 데이터 설명**
  * **연월** - `2025-01` 형태의 연월 정보입니다.
  * **대여소번호** - 각 대여소마다 부여되는 고유 번호입니다.
  * **이용건수** - 연월, 대여소번호 기준 이용건수의 합계입니다.
  * **이동거리** - 연월, 대여소번호 기준 이동거리의 합계입니다.
  * **이동시간** - 연월, 대여소번호 기준 이동시간의 합계입니다.
  * **자치구** - `OO구` 형태의 서울시 내 자치구 지역명입니다.

In [1]:
import pandas as pd
import numpy as np

import os
import glob

In [5]:
# 결과물 파일명 정의(CSV)
file_name = '따릉이_월별_이용정보_통합.csv'

# 원본 데이터셋 BASE_PATH 경로 정의
BASE_MASTER_PATH = '../서울시 따릉이 대여소 정보'
BASE_PATH = '../../original-datasets/서울시 공공데이터/월별 따릉이 이용정보_2018~2025'

#### 데이터셋 불러오기
##### `df_master`: 이전에 전처리 완료한 따릉이 대여소 마스터 DB를 불러옵니다.

In [6]:
df_master = pd.read_csv(os.path.join(BASE_MASTER_PATH, '통합_따릉이_대여소_마스터.csv'))
# 병합을 위해 대여소번호에서 숫자만 추출하여 문자열로 통일
df_master['대여소번호'] = df_master['대여소번호'].astype(str).str.extract(r'(\d+)')[0]
# 마스터 정보는 대여소번호, 자치구만 사용
df_master = df_master[['대여소번호', '자치구']]

df_master.head()

,대여소번호,자치구
0,311,중구
1,373,중구
2,444,중구
3,868,용산구
4,3535,광진구


##### 각각의 CSV/XLSX 파일에 맞게 데이터를 읽고, 
##### 정제 및 전처리를 수행하는 함수를 작성합니다.

In [10]:
# 전처리 및 압축 로직을 수행할 함수 정의
def standardize_and_aggregate(filepath):
    try:
        print(f"처리 중: {os.path.basename(filepath)} ... ", end="")
        
        # 확장자에 따른 데이터 로드 (인코딩 에러 방지)
        if filepath.endswith('.csv'):
            try:
                df = pd.read_csv(filepath, encoding='cp949') # csv(cp949)
            except UnicodeDecodeError:
                df = pd.read_csv(filepath, encoding='utf-8') # csv(utf-8)
        else:
            df = pd.read_excel(filepath)                     # xlsx
            
        # 컬럼명의 따옴표와 공백 우선 제거
        df.columns = df.columns.str.replace("'", "").str.strip()

        # 모든 문자열 데이터의 따옴표 일괄 제거
        string_cols = df.select_dtypes(include=['object', 'string']).columns
        for col in string_cols:
            df[col] = df[col].astype(str).str.replace("'", "").str.strip()
             
        # 연도별로 제각각인 컬럼명 표준화
        col_map = {
            '대여년월': '연월', '대여일자': '연월', 
            '이용거리(M)': '이동거리', '이동거리(M)': '이동거리',
            '이동시간(분)': '이용시간', '이용시간(분)': '이용시간', '이용시간(본)': '이용시간',
            '대여소': '대여소명' # 18년도 일부 파일 대응용
        }
        df.rename(columns=col_map, inplace=True)
        
        # 날짜 포맷 통일 (예: '2018-01', '201801' -> '2018-01'으로 통일)
        df['연월'] = df['연월'].astype(str).str.replace('-', '')
        df['연월'] = df['연월'].str[:4] + '-' + df['연월'].str[4:6] # 숫자만 남긴 후, 연월 포맷팅
        
        # 대여소번호 정제 (문자열 내 숫자만 추출) 및 결측치 제거
        df['대여소번호'] = df['대여소번호'].astype(str).str.extract(r'(\d+)')[0]
        df = df.dropna(subset=['대여소번호'])
        
        # 월별, 대여소번호별로 그룹화하여 데이터 압축 (이용건수, 이동거리, 이용시간 합계)
        agg_df = df.groupby(['연월', '대여소번호'])[['이용건수', '이동거리', '이용시간']].sum().reset_index()
        
        print(f"전처리 및 압축 완료. (압축 후: {len(agg_df)}행)")
        return agg_df
        
    except Exception as e:
        print(f"***에러 발생***: {e}")
        return pd.DataFrame()

##### 따릉이 이용정보 파일은 총 61개로, 파일명을 하나씩 입력하기에는 비효율적입니다.
##### ipynb 파일이 위치한 현재 폴더 경로에서 자동으로 전체 이용정보 파일명을 읽어옵니다.

In [11]:
# 모든 하위 폴더 내 이용정보 파일 자동으로 찾기 (재귀)
file_patterns = [os.path.join(BASE_PATH, "**/*이용정보(월별)*.csv"),
                os.path.join(BASE_PATH, "**/*이용정보(월별)*.xlsx")]
usage_files = []

for pattern in file_patterns:
    usage_files.extend(glob.glob(pattern, recursive=True)) # recursive=True: 하위 폴더들을 모두 탐색

# usage_files 리스트에 따릉이 이용정보 파일명들이 모두 할당됨
print(f'총 {len(usage_files)}개의 이용정보 파일을 병합합니다.')

총 61개의 이용정보 파일을 병합합니다.


##### 이전에 정의한 함수, 전체 이용정보 파일명으로 데이터 정제/전처리를 수행합니다.

In [12]:
# 반복문으로 모든 파일 전처리 후 리스트에 담기
processed_data = []
for file in sorted(usage_files):
    df_processed = standardize_and_aggregate(file)
    if not df_processed.empty:
        processed_data.append(df_processed)

processed_data

처리 중: 서울특별시 공공자전거 이용정보(월별)_18.01.csv ... 전처리 및 압축 완료. (압축 후: 1026행)
처리 중: 서울특별시 공공자전거 이용정보(월별)_18.02.csv ... 전처리 및 압축 완료. (압축 후: 1033행)
처리 중: 서울특별시 공공자전거 이용정보(월별)_18.03.csv ... 전처리 및 압축 완료. (압축 후: 1104행)
처리 중: 서울특별시 공공자전거 이용정보(월별)_18.04.csv ... 전처리 및 압축 완료. (압축 후: 1261행)
처리 중: 서울특별시 공공자전거 이용정보(월별)_18.05.csv ... 전처리 및 압축 완료. (압축 후: 1267행)
처리 중: 서울특별시 공공자전거 이용정보(월별)_18.06.csv ... 전처리 및 압축 완료. (압축 후: 1267행)
처리 중: 서울특별시 공공자전거 이용정보(월별)_18.07.xlsx ... 전처리 및 압축 완료. (압축 후: 1275행)
처리 중: 서울특별시 공공자전거 이용정보(월별)_18.08.xlsx ... 전처리 및 압축 완료. (압축 후: 1282행)
처리 중: 서울특별시 공공자전거 이용정보(월별)_18.09.xlsx ... 전처리 및 압축 완료. (압축 후: 1325행)
처리 중: 서울특별시 공공자전거 이용정보(월별)_18.10.xlsx ... 전처리 및 압축 완료. (압축 후: 1428행)
처리 중: 서울특별시 공공자전거 이용정보(월별)_18.11.xlsx ... 전처리 및 압축 완료. (압축 후: 1480행)
처리 중: 서울특별시 공공자전거 이용정보(월별)_18.12.xlsx ... 전처리 및 압축 완료. (압축 후: 1523행)
처리 중: 서울특별시 공공자전거 이용정보(월별)_19.01.xlsx ... 전처리 및 압축 완료. (압축 후: 1530행)
처리 중: 서울특별시 공공자전거 이용정보(월별)_19.02.xlsx ... 전처리 및 압축 완료. (압축 후: 1529행)
처리 중: 서울특별시 공공자전거 이용정보(월별)_19.03.xlsx ..

[           연월 대여소번호  이용건수     이동거리  이용시간
 0     2018-01  1001   169   492840  2670
 1     2018-01  1002    76   189260  1907
 2     2018-01  1003    68   138150  1373
 3     2018-01  1004   103   318910  2856
 4     2018-01  1006    43   177950   951
 ...       ...   ...   ...      ...   ...
 1021  2018-01   930   117   247140  1687
 1022  2018-01   931   359  1015240  7287
 1023  2018-01   932   133   430990  2940
 1024  2018-01   933   150   539420  3250
 1025  2018-01  9999   343   656370  3343
 
 [1026 rows x 5 columns],
            연월 대여소번호  이용건수     이동거리  이용시간
 0     2018-02  1001   196   728120  3576
 1     2018-02  1002    91   355180  1912
 2     2018-02  1003    89   411970  2030
 3     2018-02  1004    83   374440  2715
 4     2018-02  1006    61   283380  2044
 ...       ...   ...   ...      ...   ...
 1028  2018-02   930   146   489480  2676
 1029  2018-02   931   287  1196840  7015
 1030  2018-02   932   141   418080  2581
 1031  2018-02   933   142   500310  3857
 1032 

#### 데이터셋 병합
##### `df_merged`: 모든 데이터를 마스터 DB와 병합하여 필요한 데이터를 저장합니다.

In [13]:
# 모든 연도 데이터를 하나로 병합(concat)
df_usage = pd.concat(processed_data, ignore_index=True)

# 마스터 DB와 병합하여 '자치구' 정보 붙이기 (Left Join)
df_merged = pd.merge(df_usage, df_master, on='대여소번호', how='left')

# 자치구 정보가 없는 행 확인
missing_gu = df_merged['자치구'].isna().sum()
print('전처리 완료.')
print(f'총 누적 데이터: {df_merged.shape[0]}행') # len(df_merged)
print(f'자치구 매칭 실패 건수: {missing_gu}건, 전체의 {missing_gu/len(df_merged)*100:.2f}%)')

df_merged

전처리 완료.
총 누적 데이터: 215754행
자치구 매칭 실패 건수: 1276건, 전체의 0.59%)


,연월,대여소번호,이용건수,이동거리,이용시간,자치구
0,2018-01,1001,169,492840.00,2670,강동구
1,2018-01,1002,76,189260.00,1907,강동구
2,2018-01,1003,68,138150.00,1373,강동구
3,2018-01,1004,103,318910.00,2856,강동구
4,2018-01,1006,43,177950.00,951,강동구
...,...,...,...,...,...,...
215749,2025-12,992,43,167823.67,1195,은평구
215750,2025-12,993,274,380662.09,3909,은평구
215751,2025-12,994,98,198713.74,1763,은평구
215752,2025-12,995,223,495411.99,3823,은평구


### 결측치 확인 및 저장
##### 매핑 과정 중 0.59%가 누락되었으나, 이는 분석 결과에 영향을 미치지 않는 안전한 수치입니다.
##### 자치구 매핑 과정 중 누락된 0.59%는 `미상`으로 처리합니다.
##### 결과 데이터프레임을 CSV 파일로 저장합니다. 

In [14]:
# 병합 데이터프레임 결측치 확인
missing_values = df_merged.isna().sum()
print("병합 데이터프레임 결측치 현황:")
print(missing_values, '\n')

# 누락된 대여소(자치구)를 '미상'으로 채우기
df_merged['자치구'] = df_merged['자치구'].fillna('미상')
print("자치구 결측치 채운 후:", df_merged['자치구'].isna().sum())

병합 데이터프레임 결측치 현황:
연월          0
대여소번호       0
이용건수        0
이동거리        0
이용시간        0
자치구      1276
dtype: int64 

자치구 결측치 채운 후: 0


In [15]:
# 결과 데이터프레임 저장
df_merged.to_csv(file_name, index=False, encoding='utf-8-sig')
print(f"최종 통합 데이터프레임이 '{file_name}'로 저장되었습니다.")

최종 통합 데이터프레임이 '따릉이_월별_이용정보_통합.csv'로 저장되었습니다.
